# Safety-Aware Autonomous AI

### Case Study 56 · Industrial operational decision support

This runnable notebook demonstrates a conservative framework for deciding which industrial decisions may be automated, which need qualified human approval, and which must never be delegated to AI.

> **Safety boundary:** This notebook is an educational decision-support exercise. It does not connect to equipment, issue commands, authorize operational work, or replace qualified safety, legal, or engineering review.

**Decision classes**

1. `independent_automation` — bounded, validated, routine, reversible, low-consequence work.
2. `human_approval` — meaningful impact, missing context, uncertain inputs, or limited reversibility.
3. `never_delegate` — life safety, emergency response, safety barriers, or catastrophic consequence.

Assess each decision against **potential harm, reversibility, uncertainty, and regulatory impact**. The strictest dimension sets the limit.

**Run in Google Colab:** Open this notebook from the repository and run cells from top to bottom. Add an API key only if you want live model analysis; the notebook uses a conservative fallback when no key is supplied.


## 1. Install dependencies

The API key is requested securely at runtime and is never stored in this notebook. Do not paste keys into code cells or commit them to GitHub.

In [ ]:
%pip -q install --upgrade openai pydantic pandas


## 2. Define and validate the structured assessment

The response schema requires the classification, risk level, rationale, key assumptions, safeguards, and a note for each of the four risk dimensions.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

Classification = Literal[
    "independent_automation", "human_approval", "never_delegate"
]
RiskLevel = Literal["low", "moderate", "high", "critical"]

class RiskFactors(BaseModel):
    model_config = ConfigDict(extra="forbid")
    potential_harm: str = Field(min_length=2, max_length=240)
    reversibility: str = Field(min_length=2, max_length=240)
    uncertainty: str = Field(min_length=2, max_length=240)
    regulatory_impact: str = Field(min_length=2, max_length=240)

class SafetyAssessment(BaseModel):
    model_config = ConfigDict(extra="forbid")
    classification: Classification
    risk_level: RiskLevel
    rationale: str = Field(min_length=20, max_length=900)
    key_assumptions: list[str] = Field(min_length=1, max_length=5)
    human_safeguards: list[str] = Field(min_length=1, max_length=6)
    factors: RiskFactors

JSON_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "classification": {"type": "string", "enum": [
            "independent_automation", "human_approval", "never_delegate"
        ]},
        "risk_level": {"type": "string", "enum": ["low", "moderate", "high", "critical"]},
        "rationale": {"type": "string"},
        "key_assumptions": {"type": "array", "items": {"type": "string"}},
        "human_safeguards": {"type": "array", "items": {"type": "string"}},
        "factors": {
            "type": "object", "additionalProperties": False,
            "properties": {
                "potential_harm": {"type": "string"},
                "reversibility": {"type": "string"},
                "uncertainty": {"type": "string"},
                "regulatory_impact": {"type": "string"},
            },
            "required": ["potential_harm", "reversibility", "uncertainty", "regulatory_impact"],
        },
    },
    "required": ["classification", "risk_level", "rationale", "key_assumptions", "human_safeguards", "factors"],
}

print("Assessment schema ready.")


## 3. Configure the AI provider

Defaults to OpenAI `gpt-4.1-mini`. You can set `AI_PROVIDER`, `AI_MODEL`, and (for an OpenAI-compatible provider) `AI_BASE_URL` in the runtime environment. Entering no API key keeps the notebook in safe-fallback mode.

In [ ]:
import getpass
import os

AI_PROVIDER = os.getenv("AI_PROVIDER", "openai")
AI_MODEL = os.getenv("AI_MODEL", "gpt-4.1-mini")
AI_BASE_URL = os.getenv("AI_BASE_URL")
API_KEY = os.getenv("AI_API_KEY") or os.getenv("OPENAI_API_KEY")

if not API_KEY:
    try:
        API_KEY = getpass.getpass("API key (input hidden; press Enter for fallback mode): ").strip() or None
    except Exception:
        API_KEY = None

client = None
if API_KEY and (AI_PROVIDER == "openai" or AI_BASE_URL):
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=AI_BASE_URL) if AI_BASE_URL else OpenAI(api_key=API_KEY)
    print(f"Live model mode: provider={AI_PROVIDER}, model={AI_MODEL}")
else:
    print("Safe fallback mode: no supported provider/key configured. No external model call will be made.")


## 4. Safety-first classifier

The model is advisory. A deterministic high-consequence check prevents emergency, life-safety, interlock, or catastrophic signals from being downgraded by model output. If the model call fails or returns invalid data, the fallback requires human approval, or marks explicit critical signals `never_delegate`. No tools or equipment actions are available to the model.

In [ ]:
import re
from pydantic import ValidationError

CRITICAL_SIGNAL = re.compile(
    r"fatal|life.?threat|bypass|disable.*(alarm|interlock|guard)|"
    r"silenc(e|ing).*alarm|alarm.*silenc|over.?pressure|toxic release|"
    r"chlorine leak|explosion|fire|emergency stop|safety system|evacuat",
    re.IGNORECASE,
)

SYSTEM_INSTRUCTIONS = (
    "You are an industrial safety decision-support assistant. Classify conservatively using "
    "potential harm, reversibility, uncertainty, and regulatory impact. Never delegate life-safety, "
    "emergency response, safety barrier, interlock bypass, or potentially catastrophic decisions. "
    "Require qualified human approval when context is missing or consequences are material. "
    "Independent automation is limited to low-consequence, bounded, validated, reversible routine tasks. "
    "State assumptions and safeguards. Advisory only; never direct equipment actions."
)

def safe_fallback(decision: str, context: str = "") -> SafetyAssessment:
    critical = bool(CRITICAL_SIGNAL.search(f"{decision} {context}"))
    return SafetyAssessment(
        classification="never_delegate" if critical else "human_approval",
        risk_level="critical" if critical else "high",
        rationale=(
            "A high-consequence safety signal is present. Keep this decision under qualified human "
            "authority and engineered safety-system control." if critical else
            "A validated model assessment is unavailable. Treat this operational decision as high risk "
            "and require qualified human review before any action."
        ),
        key_assumptions=["The description may omit site hazards, interlocks, permits, or mandatory procedures."],
        human_safeguards=[
            "Pause and consult the site procedure and a qualified operations or safety lead.",
            "Do not connect this advisory result to equipment or control systems.",
        ],
        factors=RiskFactors(
            potential_harm="Unknown; assume significant until assessed.",
            reversibility="Unknown; confirm safe recovery before proceeding.",
            uncertainty="High because validated operating limits and site context are missing.",
            regulatory_impact="Unknown; verify applicable permits, standards, and procedures.",
        ),
    )

def assess_decision(decision: str, context: str = "") -> tuple[SafetyAssessment, str]:
    if not isinstance(decision, str) or len(decision.strip()) < 12:
        raise ValueError("Describe the decision in at least 12 characters.")
    if len(decision) > 1600 or len(context) > 700:
        raise ValueError("Decision or context exceeds the notebook limits.")

    if client is None:
        result, source = safe_fallback(decision, context), "safe fallback"
    else:
        try:
            response = client.responses.create(
                model=AI_MODEL,
                instructions=SYSTEM_INSTRUCTIONS,
                input=f"Decision: {decision}\nSite context: {context or 'Not provided'}",
                text={"format": {
                    "type": "json_schema", "name": "industrial_safety_assessment",
                    "strict": True, "schema": JSON_SCHEMA,
                }},
            )
            result = SafetyAssessment.model_validate_json(response.output_text)
            source = "model (advisory)"
        except Exception as exc:
            result, source = safe_fallback(decision, context), f"safe fallback ({type(exc).__name__})"

    # Deterministic override: critical hazards cannot be classified as automatable.
    if CRITICAL_SIGNAL.search(f"{decision} {context}") and result.classification != "never_delegate":
        result = safe_fallback(decision, context)
        source = "critical-signal safety override"
    return result, source

print("Classifier ready. No equipment-control tools are defined.")


## 5. Review industrial examples

These are discussion examples, not site-specific operating instructions. Compare the reference decision with the AI advisory output and its assumptions.

In [ ]:
import pandas as pd
from IPython.display import display

EXAMPLES = [
    {
        "label": "Bounded, reversible adjustment",
        "decision": "Adjust a non-critical pump setpoint within its validated approved band and alert the operator.",
        "context": "Routine operation; tested limits, independent interlocks, and rollback are in place.",
        "reference_class": "Safe to automate independently",
    },
    {
        "label": "Critical maintenance trade-off",
        "decision": "Postpone an inspection on a high-pressure vessel because production demand is high.",
        "context": "The inspection interval and asset integrity are governed by site procedures and applicable requirements.",
        "reference_class": "Requires human approval",
    },
    {
        "label": "Emergency and life safety",
        "decision": "Silence the alarm during a possible chlorine leak if the sensor appears faulty.",
        "context": "Possible toxic release; the decision could affect evacuation and emergency response.",
        "reference_class": "Must never be delegated to AI",
    },
]

reference = pd.DataFrame([{k: row[k] for k in ("label", "reference_class")} for row in EXAMPLES])
display(reference)

for example in EXAMPLES:
    result, source = assess_decision(example["decision"], example["context"])
    print(f"\n### {example['label']} · {source}")
    display(pd.Series(result.model_dump()))


## 6. Try your own decision

Edit the two variables below, then run the cell. Include the consequence of failure, validated operating limits, reversibility, and relevant procedures when known. Missing context increases uncertainty.

In [ ]:
decision_to_review = "Describe the industrial decision the AI is being asked to make here."
site_context = "Optional: equipment, safe operating envelope, failure consequence, and applicable procedures."

if decision_to_review.startswith("Describe the industrial decision"):
    print("Edit decision_to_review and site_context above, then rerun this cell.")
else:
    assessment, source = assess_decision(decision_to_review, site_context)
    print("Assessment source:", source)
    display(pd.Series(assessment.model_dump()))


## 7. Operating guardrails and shutdown checklist

- **Constrain:** allowlist actions and operating ranges; deny by default; separate read, recommend, approve, and execute permissions.
- **Protect:** no model access to emergency stops, interlocks, safety instrumented systems, or direct equipment controls.
- **Monitor:** record model/version, inputs, outputs, human approvals, decisions, overrides, incidents, and outcomes under site retention rules.
- **Escalate:** route anomalies, stale sensors, out-of-envelope requests, or uncertain outputs to the control room and qualified personnel.
- **Shut down:** revoke the agent's permissions, isolate it from operational networks, transfer authority to trained staff, preserve logs, and follow the site's emergency procedure. Only qualified staff restore service.

> Model output is advisory only. It never authorizes work or replaces qualified safety, legal, or engineering review.
